# Train a six-class CNN
Upload the project ZIP and the original `fer2013.csv` with emotion, pixels and Usage columns. The notebook runs `train.py` and `evaluate.py`. Training uses Training; early stopping uses PublicTest; final evaluation uses PrivateTest. Disgust is excluded.
In Colab choose Runtime > Change runtime type > GPU if available, then run the cells in order.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, subprocess, sys
uploaded = files.upload()  # select BOTH project ZIP and original fer2013.csv
archives = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
csvs = [Path(name).resolve() for name in uploaded if name.lower().endswith('.csv')]
assert len(archives) == 1 and len(csvs) == 1, 'Upload exactly one project ZIP and one FER CSV.'
csv_path = csvs[0]
destination = Path('/content/minor_project')
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for entry in archive.infolist():
        target = (destination / entry.filename).resolve()
        assert target.is_relative_to(destination.resolve()), 'Unsafe ZIP path.'
    archive.extractall(destination)
projects = list(destination.rglob('train.py'))
assert len(projects) == 1, 'Expected one project containing train.py.'
os.chdir(projects[0].parent)
print('Project:', Path.cwd(), '\nDataset:', csv_path)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)
# If Colab asks for a runtime restart after installing, restart and rerun the upload cell.
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Change 30 only if you can explain the experiment. Early stopping may finish sooner.
subprocess.run([sys.executable, 'train.py', '--csv', str(csv_path), '--epochs', '30'], check=True)

In [ ]:
# Evaluate only after training choices are final. Do not tune repeatedly on PrivateTest.
subprocess.run([sys.executable, 'evaluate.py', '--csv', str(csv_path)], check=True)
import json
results = json.loads(Path('reports/evaluation.json').read_text())
print('Actual test accuracy:', round(results['accuracy'] * 100, 2), '%')
print('Actual macro F1:', round(results['macro_f1'], 4))

In [ ]:
output = Path('/content/trained_cnn_and_reports.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('models/emotion_model.keras', 'models/emotion_model.json'):
        archive.write(name, name)
    for item in Path('reports').glob('*'):
        if item.is_file(): archive.write(item, str(item))
files.download(str(output))
print('Extract into your laptop project, install requirements-training.txt, and restart app.py.')